# Ollama on Google Colab T4
**Generation + Embedding** with OpenAI-compatible API via Ngrok

| Model | Purpose | Dims |
|---|---|---|
| `gemma2:9b-instruct-q5_0` | Generation | - |
| `nomic-embed-text` | Embedding (multilingual) | 768 |

> bge-m3 has a NaN bug on Ollama 0.22.1 — using nomic-embed-text instead

> Run cells top to bottom

## Step 1 - Install zstd + Ollama

In [ ]:
# zstd required by Ollama installer
# lshw + pciutils required for GPU detection
!apt-get install -y zstd lshw pciutils

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Verify installation
import subprocess
r = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
print('Ollama installed:', r.stdout.strip())

## Step 2 - Define models

In [ ]:
GENERATION_MODEL = "gemma2:9b-instruct-q5_0"
EMBEDDING_MODEL  = "nomic-embed-text"
EMBEDDING_SIZE   = 768

print(f"Generation : {GENERATION_MODEL}")
print(f"Embedding  : {EMBEDDING_MODEL} ({EMBEDDING_SIZE} dims)")

## Step 3 - Start Ollama server (stays alive between cells)

In [ ]:
import subprocess, time, os, urllib.request

# Kill any old ollama processes
subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(2)

# Use subprocess.Popen - stays alive across ALL cells
# Never use !nohup ... & - it dies when the cell finishes
server = subprocess.Popen(
    ['ollama', 'serve'],
    env={**os.environ, 'OLLAMA_HOST': '0.0.0.0:11434', 'OLLAMA_ORIGIN': '*'},
    stdout=open('/content/ollama.log', 'w'),
    stderr=subprocess.STDOUT
)

print(f'Server PID: {server.pid}')
print('Waiting for server to start...')
time.sleep(6)

# Verify server is reachable
try:
    urllib.request.urlopen('http://localhost:11434')
    print('Server is running on port 11434!')
except Exception as e:
    print(f'Server error: {e}')
    print(open('/content/ollama.log').read())

# Check GPU detected in log
time.sleep(2)
log = open('/content/ollama.log').read()
if 'Tesla T4' in log:
    print('Tesla T4 GPU detected!')
else:
    print('GPU not in log yet - will appear after first model loads')

## Step 4 - Pull models

In [ ]:
import subprocess

def pull_model(model_name):
    print(f'Pulling {model_name} ... (may take a few minutes)')
    result = subprocess.run(
        ['ollama', 'pull', model_name],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'{model_name} ready!')
    else:
        print(f'Failed to pull {model_name}: {result.stderr}')

pull_model(GENERATION_MODEL)
pull_model(EMBEDDING_MODEL)

# List all available models
print('\nAvailable models:')
r = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print(r.stdout)

## Step 5 - Test generation + embedding locally

In [ ]:
!pip install openai -q

from openai import OpenAI
import math

# Local client - same OpenAI structure, different base_url
client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama'  # required by SDK but ignored by Ollama
)

# --- Test Generation ---
print('Generation test:')
resp = client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[{'role': 'user', 'content': 'ما عاصمة مصر؟'}]
)
print(resp.choices[0].message.content)

# --- Test Embedding ---
print('\nEmbedding test:')
texts = [
    'مرحبا بالعربية',
    'Hello in English',
    'Bonjour en francais',
    'مصر بلد جميل'
]

emb_resp = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=texts
)

all_ok = True
for i, item in enumerate(emb_resp.data):
    vec = item.embedding
    has_nan = any(math.isnan(v) for v in vec)
    status = 'NaN ERROR' if has_nan else 'OK'
    if has_nan:
        all_ok = False
    print(f"  [{status}] '{texts[i]}' -> {len(vec)} dims")

if all_ok:
    print(f'\nAll embeddings clean! EMBEDDING_MODEL_SIZE={len(emb_resp.data[0].embedding)}')
else:
    print('\nNaN detected - switch embedding model!')

## Step 6 - Expose via Ngrok

In [ ]:
!pip install pyngrok -q

from google.colab import userdata
from pyngrok import ngrok, conf

# Store your ngrok token in Colab Secrets with key: NGROK_AUTHTOKEN
ngrok_auth = userdata.get('NGROK_AUTHTOKEN')
conf.get_default().auth_token = ngrok_auth

# Kill old tunnels
ngrok.kill()

# Port 11434 - must match the server port above
public_url = ngrok.connect(11434).public_url

print(f'Public URL: {public_url}')
print(f'\nCopy this to your .env file:')
print(f'OPENAI_API_KEY="ollama"')
print(f'OPENAI_API_URL="{public_url}/v1"')
print(f'GENERATION_MODEL_ID="{GENERATION_MODEL}"')
print(f'EMBEDDING_MODEL_ID="{EMBEDDING_MODEL}"')
print(f'EMBEDDING_MODEL_SIZE={EMBEDDING_SIZE}')

## Step 7 - Test via Ngrok (external access)

In [ ]:
import math
from openai import OpenAI

# This is how your external app connects
external_client = OpenAI(
    base_url=f'{public_url}/v1',
    api_key='ollama'
)

# Test generation via ngrok
print('External generation test (via Ngrok):')
resp = external_client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[{'role': 'user', 'content': 'قل مرحبا باختصار'}]
)
print(resp.choices[0].message.content)

# Test embedding via ngrok
print('\nExternal embedding test (via Ngrok):')
emb = external_client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=['اختبار التضمين عبر الرابط العام']
)
vec = emb.data[0].embedding
has_nan = any(math.isnan(v) for v in vec)
print(f"Result: {'NaN ERROR' if has_nan else 'Clean'} | dims={len(vec)}")
if not has_nan:
    print('Setup complete! Your app can now use the ngrok URL.')

---
## Final .env Config

```dotenv
OPENAI_API_KEY="ollama"
OPENAI_API_URL="https://YOUR-NGROK-URL.ngrok-free.app/v1"
COHERE_API_KEY=""

GENERATION_MODEL_ID="gemma2:9b-instruct-q5_0"
EMBEDDING_MODEL_ID="nomic-embed-text"
EMBEDDING_MODEL_SIZE=768

INPUT_DAFAULT_MAX_CHARACTERS=1024
GENERATION_DAFAULT_MAX_TOKENS=200
GENERATION_DAFAULT_TEMPERATURE=0.1

VECTOR_DB_BACKEND="QDRANT"
VECTOR_DB_PATH="qdrant_db"
VECTOR_DB_DISTANCE_METHOD="cosine"

PRIMARY_LANG = "ar"
DEFAULT_LANG = "ar"
```

IMPORTANT NOTES:
- Every time you restart Colab, update OPENAI_API_URL with the new ngrok URL
- If you had an old Qdrant DB (from Cohere=384 dims or bge-m3=1024 dims), DELETE it and re-index. nomic-embed-text uses 768 dims and sizes must match.